# Credit Card Fraud Detection using Logistic Regression

In this project, Logistic Regression is used to predict whether a credit card transaction is fraudulent or not.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## Load Dataset

In [2]:
df = pd.read_csv("credit_card_fraud_10k.csv")

df.head()

,transaction_id,amount,transaction_hour,merchant_category,foreign_transaction,location_mismatch,device_trust_score,velocity_last_24h,cardholder_age,is_fraud
0,1,84.47,22,Electronics,0,0,66,3,40,0
1,2,541.82,3,Travel,1,0,87,1,64,0
2,3,237.01,17,Grocery,0,0,49,1,61,0
3,4,164.33,4,Grocery,0,1,72,3,34,0
4,5,30.53,15,Food,0,0,79,0,44,0


In [3]:
print("Dataset Shape:", df.shape)

Dataset Shape: (10000, 10)


In [4]:
print("\nColumns:")
print(df.columns.tolist())


Columns:
['transaction_id', 'amount', 'transaction_hour', 'merchant_category', 'foreign_transaction', 'location_mismatch', 'device_trust_score', 'velocity_last_24h', 'cardholder_age', 'is_fraud']


In [5]:
print("\nData Types:")
print(df.dtypes)


Data Types:
transaction_id           int64
amount                 float64
transaction_hour         int64
merchant_category          str
foreign_transaction      int64
location_mismatch        int64
device_trust_score       int64
velocity_last_24h        int64
cardholder_age           int64
is_fraud                 int64
dtype: object


In [6]:
print("\nMissing Values:")
print(df.isnull().sum())


Missing Values:
transaction_id         0
amount                 0
transaction_hour       0
merchant_category      0
foreign_transaction    0
location_mismatch      0
device_trust_score     0
velocity_last_24h      0
cardholder_age         0
is_fraud               0
dtype: int64


## Check Target Variable

The target variable is `is_fraud`.
0 represents a normal transaction and 1 represents a fraudulent transaction.

In [7]:
print(df["is_fraud"].value_counts())

is_fraud
0    9849
1     151
Name: count, dtype: int64


In [8]:
print(df["is_fraud"].value_counts(normalize=True) * 100)

is_fraud
0    98.49
1     1.51
Name: proportion, dtype: float64


In [9]:
X = df.drop(columns=["is_fraud", "transaction_id"])
y = df["is_fraud"]

In [10]:
print("Features:")
print(X.columns.tolist())

Features:
['amount', 'transaction_hour', 'merchant_category', 'foreign_transaction', 'location_mismatch', 'device_trust_score', 'velocity_last_24h', 'cardholder_age']


In [11]:
print("\nTarget:")
print(y.name)


Target:
is_fraud


In [12]:
categorical_features = ["merchant_category"]

numerical_features = [
    "amount",
    "transaction_hour",
    "foreign_transaction",
    "location_mismatch",
    "device_trust_score",
    "velocity_last_24h",
    "cardholder_age"
]

## Split Dataset

The dataset is divided into training and testing data.
80% is used for training and 20% is used for testing.

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [14]:
print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (8000, 8)
Testing data: (2000, 8)


In [15]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

## Build Logistic Regression Model

In [16]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

In [17]:
model.fit(X_train, y_train)

print("Model training completed.")

Model training completed.


## Make Predictions

The trained model is used to predict fraud on the testing data.

In [18]:
y_pred = model.predict(X_test)

print(y_pred[:20])

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [19]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("Accuracy Percentage:", accuracy * 100, "%")

Accuracy: 0.991
Accuracy Percentage: 99.1 %


In [20]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      1.00      1.00      1970
           1       0.75      0.60      0.67        30

    accuracy                           0.99      2000
   macro avg       0.87      0.80      0.83      2000
weighted avg       0.99      0.99      0.99      2000



In [21]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[1964    6]
 [  12   18]]


In [22]:
sample_transaction = pd.DataFrame({
    "amount": [250.50],
    "transaction_hour": [14],
    "merchant_category": ["Grocery"],
    "foreign_transaction": [0],
    "location_mismatch": [0],
    "device_trust_score": [80],
    "velocity_last_24h": [2],
    "cardholder_age": [35]
})

prediction = model.predict(sample_transaction)[0]

if prediction == 1:
    print("Prediction: Fraudulent Transaction")
else:
    print("Prediction: Normal Transaction")

Prediction: Normal Transaction


In [23]:
probability = model.predict_proba(sample_transaction)[0][1]

print("Fraud Probability:", round(probability * 100, 2), "%")

Fraud Probability: 0.0 %


In [24]:
def predict_fraud(
    amount,
    transaction_hour,
    merchant_category,
    foreign_transaction,
    location_mismatch,
    device_trust_score,
    velocity_last_24h,
    cardholder_age
):
    input_data = pd.DataFrame({
        "amount": [amount],
        "transaction_hour": [transaction_hour],
        "merchant_category": [merchant_category],
        "foreign_transaction": [foreign_transaction],
        "location_mismatch": [location_mismatch],
        "device_trust_score": [device_trust_score],
        "velocity_last_24h": [velocity_last_24h],
        "cardholder_age": [cardholder_age]
    })

    prediction = model.predict(input_data)[0]
    probability = model.predict_proba(input_data)[0][1]

    if prediction == 1:
        result = "Fraudulent Transaction"
    else:
        result = "Normal Transaction"

    return result, round(probability * 100, 2)

In [25]:
result, probability = predict_fraud(
    250.50,
    14,
    "Grocery",
    0,
    0,
    80,
    2,
    35
)

print("Prediction:", result)
print("Fraud Probability:", probability, "%")

Prediction: Normal Transaction
Fraud Probability: 0.0 %


## Gradio User Interface

Gradio is used to create a simple web interface for our trained Logistic Regression model.

In [26]:
import gradio as gr

In [27]:
gradio_app = gr.Interface(
    fn=predict_fraud,
    inputs=[
        gr.Number(label="Transaction Amount"),
        gr.Number(label="Transaction Hour"),
        gr.Dropdown(
            choices=["Electronics", "Travel", "Grocery", "Food", "Entertainment"],
            label="Merchant Category"
        ),
        gr.Radio([0, 1], label="Foreign Transaction"),
        gr.Radio([0, 1], label="Location Mismatch"),
        gr.Number(label="Device Trust Score"),
        gr.Number(label="Velocity Last 24 Hours"),
        gr.Number(label="Cardholder Age")
    ],
    outputs=[
        gr.Textbox(label="Prediction"),
        gr.Number(label="Fraud Probability (%)")
    ],
    title="Credit Card Fraud Detection",
    description="Enter transaction details to predict whether the transaction is fraudulent."
)

In [28]:
gradio_app.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Flask User Interface

In [29]:
from flask import Flask, request, render_template_string

In [30]:
app = Flask(__name__)

In [31]:
html = """
<!DOCTYPE html>
<html>
<head>
    <title>Credit Card Fraud Detection</title>
</head>
<body>

    <h1>Credit Card Fraud Detection</h1>

    <form method="POST">

        <label>Transaction Amount:</label>
        <input type="number" step="0.01" name="amount" required><br><br>

        <label>Transaction Hour:</label>
        <input type="number" name="transaction_hour" required><br><br>

        <label>Merchant Category:</label>
        <select name="merchant_category">
            <option>Electronics</option>
            <option>Travel</option>
            <option>Grocery</option>
            <option>Food</option>
            <option>Entertainment</option>
        </select><br><br>

        <label>Foreign Transaction:</label>
        <select name="foreign_transaction">
            <option value="0">No</option>
            <option value="1">Yes</option>
        </select><br><br>

        <label>Location Mismatch:</label>
        <select name="location_mismatch">
            <option value="0">No</option>
            <option value="1">Yes</option>
        </select><br><br>

        <label>Device Trust Score:</label>
        <input type="number" name="device_trust_score" required><br><br>

        <label>Velocity Last 24 Hours:</label>
        <input type="number" name="velocity_last_24h" required><br><br>

        <label>Cardholder Age:</label>
        <input type="number" name="cardholder_age" required><br><br>

        <button type="submit">Predict</button>

    </form>

    {% if result %}
        <h2>Prediction: {{ result }}</h2>
        <h3>Fraud Probability: {{ probability }}%</h3>
    {% endif %}

</body>
</html>
"""

In [32]:
@app.route("/", methods=["GET", "POST"])
def home():

    result = None
    probability = None

    if request.method == "POST":

        amount = float(request.form["amount"])
        transaction_hour = int(request.form["transaction_hour"])
        merchant_category = request.form["merchant_category"]
        foreign_transaction = int(request.form["foreign_transaction"])
        location_mismatch = int(request.form["location_mismatch"])
        device_trust_score = int(request.form["device_trust_score"])
        velocity_last_24h = int(request.form["velocity_last_24h"])
        cardholder_age = int(request.form["cardholder_age"])

        result, probability = predict_fraud(
            amount,
            transaction_hour,
            merchant_category,
            foreign_transaction,
            location_mismatch,
            device_trust_score,
            velocity_last_24h,
            cardholder_age
        )

    return render_template_string(
        html,
        result=result,
        probability=probability
    )

In [ ]:
app.run(port=5000)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [13/Oct/2026 21:12:04] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [13/Oct/2026 21:12:04] "GET /favicon.ico HTTP/1.1" 404 -
